# Prepare GSS 2004 — Likert recoding and dataset construction

Python translation of `1_prepare.Rmd`. Loads `data/source/GSS2004.dta`, recodes the four Likert batteries (Q2/Q3/Q4/Q5) into paired `_n`/`_f` columns, builds the standard control set, kNN-imputes residual NAs, and writes per-approach parquets to `data/`.

Set `approach` to:

- `'replic'` — strict B&D recoding: "can't choose" → NA. Writes `data2004_830.parquet` (listwise) and `data2004_1077.parquet` (≤10 NA outcomes + complete controls), each with a pre-imputation (`_ni`) variant.
- `'own'` — "can't choose" → neutral midpoint of the Likert. Writes `data2004_1215.parquet` (full Identity module respondents minus id 769).

Notes on fidelity vs the R source:

- R's `VIM::kNN` uses Gower distance over mixed numeric/factor columns. `sklearn.impute.KNNImputer` is numeric-only, so factor controls are ordinal-encoded before imputation and decoded back via nearest integer. Outputs match on the `_n` columns used downstream by the clustering, but factor-control imputations may differ on a handful of rows.
- `labelled::set_variable_labels` has no pandas equivalent; variable labels are reconstructed inline in `2_desc_PCA.ipynb` via a `LABELS` dict.
- The `_f` reverse on `amshamed` follows R: numeric values are flipped (`6 - x`) and the factor levels are reordered without relabelling per-row values — so `_n` and `_f` deliberately diverge for `amshamed` (`_n` is on the patriotism scale, `_f` keeps the original agree/disagree wording).

In [1]:
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
from sklearn.impute import KNNImputer

## Parameters

In [2]:
approach = 'replic'   # 'replic' or 'own'

DATA_PATH = '../data/GSS2004.dta'
OUT_DIR = '../data'

In [3]:
data2004 = pd.read_stata(DATA_PATH, convert_categoricals=False)

# `hhtype` (and possibly other GSS housekeeping columns) have duplicate value
# labels in the .dta, which pandas refuses to auto-convert. Restrict the
# labelled read to the outcome columns we actually match strings against.
OUTCOME_COLS = [
    'clseusa', 'clsetown', 'clsestat', 'clsenoam',
    'ambornin', 'amcit', 'amlived', 'amenglsh', 'amchrstn', 'amgovt', 'amfeel', 'amancstr',
    'amcitizn', 'amshamed', 'belikeus', 'ambetter', 'ifwrong', 'amsports', 'lessprd',
    'proudsss', 'proudgrp', 'proudpol', 'prouddem', 'proudeco',
    'proudspt', 'proudart', 'proudhis', 'proudmil', 'proudsci',
]
outcome_cols = [c for c in OUTCOME_COLS if c in data2004.columns]
data2004_lbl = pd.read_stata(DATA_PATH, convert_categoricals=True, columns=outcome_cols)
print(f'shape: {data2004.shape}')

shape: (2812, 1210)


## Outcome recoding

Likert items return mixed numeric/labelled values from haven; under `pd.read_stata(convert_categoricals=True)` the categories preserve the .dta value labels. We map "can't choose" either to NA (`replic`) or to the scale midpoint (`own`), producing `_n` (numeric Likert 1–5) and `_f` (factor) versions per item.

In [4]:
if approach == 'replic':
    mods = [1, 2, np.nan, 3, 4]
    mods_q4 = [1, 2, 3, 4, 5, np.nan]
    lbls = ['Not at all', 'Not very much', 'Somewhat / fairly', 'Very much']
    lbls_q4 = ['Not at all', 'Not very much', 'Neutral', 'Somewhat / fairly', 'Very much']
elif approach == 'own':
    mods = [1, 2, 3, 4, 5]
    mods_q4 = [1, 2, 3, 4, 5, 3]
    lbls = ['Not at all', 'Not very much', "Can't choose", 'Somewhat / fairly', 'Very much']
    lbls_q4 = lbls + ["Can't choose"]
else:
    raise ValueError(f'Unknown approach: {approach}')

In [5]:
def to_factor(numeric_col, mods, lbls):
    """R-style `factor()` with NA-stripped levels and positional labelling.

    R drops NA entries from `levels` *before* aligning `labels` positionally, so a
    mid-sequence NA in `mods` (e.g. replic's [1, 2, NA, 3, 4]) must not shift the
    remaining labels. Stripping NA only after zipping would drop the last level
    (here `4` = top Likert category) — sending every top answer to NaN.
    Duplicate `mods` values keep the first label — matching R's behaviour for own/Q4.
    """
    levels = [m for m in mods if not pd.isna(m)]
    mapping = {}
    for m, l in zip(levels, lbls):
        if m not in mapping:
            mapping[m] = l
    ordered = list(mapping.values())
    return pd.Categorical(numeric_col.map(mapping), categories=ordered, ordered=False)


def recode_battery(df, df_lbl, vars_, recode_map, mods, lbls):
    for v in vars_:
        s = df_lbl[v].astype(object)
        df[v + '_n'] = s.map(recode_map).astype(float)
        df[v + '_f'] = to_factor(df[v + '_n'], mods, lbls)
    return df

### Q.2 — closeness

In [6]:
q2_vars = ['clseusa']
recode_q2 = {
    'VERY CLOSE':       mods[4],
    'close':            mods[3],
    'CANT CHOOSE':      mods[2],
    'NOT VERY CLOSE':   mods[1],
    'NOT CLOSE AT ALL': mods[0],
}
data2004 = recode_battery(data2004, data2004_lbl, q2_vars, recode_q2, mods, lbls)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1133446242.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[v + '_n'] = s.map(recode_map).astype(float)
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1133446242.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[v + '_f'] = to_factor(df[v + '_n'], mods, lbls)


### Q.3 — importance

In [7]:
q3_vars = ['ambornin', 'amcit', 'amlived', 'amenglsh', 'amchrstn', 'amgovt', 'amfeel']
recode_q3 = {
    'VERY IMPORTANT':       mods[4],
    'FAIRLY IMPORTANT':     mods[3],
    'CANT CHOOSE':          mods[2],
    'NOT VERY IMPORTANT':   mods[1],
    'NOT IMPORTANT AT ALL': mods[0],
}
data2004 = recode_battery(data2004, data2004_lbl, q3_vars, recode_q3, mods, lbls)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1133446242.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[v + '_n'] = s.map(recode_map).astype(float)
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1133446242.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[v + '_f'] = to_factor(df[v + '_n'], mods, lbls)
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1133446242.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fra

### Q.4 — agreement

Uses `mods_q4` / `lbls_q4` (5-point scale incl. an explicit neutral). `amshamed` is then reversed so that high numeric values track patriotism rather than agreement-with-being-ashamed.

In [8]:
q4_vars = ['amcitizn', 'amshamed', 'belikeus', 'ambetter', 'ifwrong']
recode_q4 = {
    'STRONGLY AGREE':             mods_q4[4],
    'agree':                      mods_q4[3],
    'NEITHER AGREE NOR DISAGREE': mods_q4[2],
    'disagree':                   mods_q4[1],
    'STRONGLY DISAGREE':          mods_q4[0],
    'CANT CHOOSE':                mods_q4[5],
}
data2004 = recode_battery(data2004, data2004_lbl, q4_vars, recode_q4, mods_q4, lbls_q4)

# Reverse-code amshamed: numeric flip, factor level reorder (per-row label preserved — see notebook intro).
data2004['amshamed_n'] = 6 - data2004['amshamed_n']
rev_levels = list(data2004['amshamed_f'].cat.categories[::-1])
data2004['amshamed_f'] = data2004['amshamed_f'].cat.reorder_categories(rev_levels, ordered=False)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1133446242.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[v + '_n'] = s.map(recode_map).astype(float)
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1133446242.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[v + '_f'] = to_factor(df[v + '_n'], mods, lbls)
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1133446242.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fra

### Q.5 — pride

In [9]:
q5_vars = ['proudsss', 'proudgrp', 'proudpol', 'prouddem', 'proudeco',
           'proudspt', 'proudart', 'proudhis', 'proudmil', 'proudsci']
recode_q5 = {
    'VERY PROUD':       mods[4],
    'SOMEWHAT PROUD':   mods[3],
    'CANT CHOOSE':      mods[2],
    'NOT VERY PROUD':   mods[1],
    'NOT PROUD AT ALL': mods[0],
}
data2004 = recode_battery(data2004, data2004_lbl, q5_vars, recode_q5, mods, lbls)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1133446242.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[v + '_n'] = s.map(recode_map).astype(float)
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1133446242.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[v + '_f'] = to_factor(df[v + '_n'], mods, lbls)
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1133446242.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fra

## Survey weights

In [10]:
data2004['wgt'] = pd.to_numeric(data2004['wtssnr'], errors='coerce')

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1453980270.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['wgt'] = pd.to_numeric(data2004['wtssnr'], errors='coerce')


## Controls

Numeric comparisons against the raw GSS codes (matching the R, which works through `haven_labelled` values rather than the labelled strings).

### Age

In [11]:
data2004['age_n'] = pd.to_numeric(data2004['age'], errors='coerce')
data2004['age_f'] = pd.cut(
    data2004['age_n'],
    bins=[-np.inf, 25, 40, 55, 70, np.inf],
    right=False,
    labels=['< 25 yo', '25-40 yo', '40-55 yo', '55-70 yo', '>70 yo'],
)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/720226053.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['age_n'] = pd.to_numeric(data2004['age'], errors='coerce')
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/720226053.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['age_f'] = pd.cut(


### Sex

In [12]:
sex_code = pd.Series(np.where(data2004['sex'] == 1, 1,
                     np.where(data2004['sex'] == 2, 0, np.nan)),
                     index=data2004.index)
data2004['sex_f'] = pd.Categorical(
    sex_code.map({0.0: 'F', 1.0: 'M'}),
    categories=['F', 'M'], ordered=False,
)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/3415418091.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['sex_f'] = pd.Categorical(


### Race

Hispanic ethnicity (`ethnic` in 17/22/38) takes precedence over the 3-level `race` variable — reproducing B&D's choice.

In [13]:
race = data2004['race']
ethnic = data2004['ethnic']
race_code = pd.Series(np.where(
    ethnic.isin([17, 22, 38]), 3,
    np.where(race == 1, 1,
    np.where(race == 2, 2,
    np.where(race == 3, 4, np.nan)))), index=data2004.index)
data2004['race_f'] = pd.Categorical(
    race_code.map({1.0: 'White', 2.0: 'Black', 3.0: 'Hispanic', 4.0: 'Other race'}),
    categories=['White', 'Black', 'Hispanic', 'Other race'], ordered=False,
)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1226794743.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['race_f'] = pd.Categorical(


### Education

In [14]:
data2004['educ_n'] = pd.to_numeric(data2004['educ'], errors='coerce')
data2004['educ_f'] = pd.cut(
    data2004['educ_n'],
    bins=[-np.inf, 12, 16, 17, np.inf],
    right=False,
    labels=['< High school', 'High school or some college', 'Bachelor', 'Advanced degree'],
)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/3813326253.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['educ_n'] = pd.to_numeric(data2004['educ'], errors='coerce')
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/3813326253.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['educ_f'] = pd.cut(


### Religion (tradition)

In [15]:
denom  = data2004['denom']
race_n = data2004['race']
other_ = data2004['other']
relig  = data2004['relig']
attend = data2004['attend']

black_prot = (
    denom.isin([12, 13, 20, 21]) |
    (denom.isin([10, 11, 14, 15, 18, 23, 28]) & (race_n == 2)) |
    other_.isin([7, 14, 15, 21, 37, 38, 56, 78, 79, 85, 86, 87, 88, 98,
                 103, 104, 128, 133]) |
    ((other_ == 93) & (race_n == 2))
)
evang_prot = (
    (denom.isin([10, 14, 15, 18, 23]) & (race_n != 2)) |
    denom.isin([32, 33, 34, 42]) |
    other_.isin([
        2, 3, 5, 6, 9, 10, 12, 13, 16, 18, 20, 22, 24, 26, 27,
        28, 29, 31, 32, 34, 35, 36, 39, 41, 42, 43, 45, 47, 51, 52,
        53, 55, 57, 63, 65, 66, 67, 68, 69, 76, 77, 83, 84, 90, 91,
        92, 94, 97, 100, 101, 102, 106, 107, 108, 109, 110, 111,
        112, 115, 116, 117, 118, 120, 121, 122, 124, 125, 127,
        129, 131, 132, 133, 134, 135, 138, 139, 140, 146,
    ]) |
    ((other_ == 93) & (race_n != 2)) |
    ((denom == 70) & (attend >= 4) & (attend != 9))
)
mainline_prot = (
    (denom.isin([11, 28]) & (race_n != 2)) |
    denom.isin([22, 30, 31, 35, 38, 40, 41, 43, 48, 50]) |
    other_.isin([1, 8, 19, 23, 25, 40, 44, 46, 48, 49, 50, 54,
                 70, 71, 72, 73, 81, 89, 96, 99, 105, 119, 148])
)
other_relig_list = other_.isin([
    11, 17, 29, 30, 33, 58, 59, 60, 61, 62, 64, 74,
    75, 80, 82, 95, 113, 114, 130, 136, 141, 145,
])

reltrad_code = pd.Series(np.nan, index=data2004.index)
reltrad_code[black_prot] = 3
reltrad_code[reltrad_code.isna() & evang_prot] = 2
reltrad_code[reltrad_code.isna() & mainline_prot] = 1
reltrad_code[reltrad_code.isna() & other_relig_list] = 7
reltrad_code[reltrad_code.isna() & ((relig == 2) | (other_ == 123))] = 4
reltrad_code[reltrad_code.isna() & (relig == 3)] = 5
reltrad_code[reltrad_code.isna() & (relig == 4)] = 6
reltrad_code[reltrad_code.isna() & (relig == 5)] = 7
reltrad_code = reltrad_code.fillna(7)  # R's `TRUE ~ 7` default

reltrad_levels = ['Mainline Protestant', 'Evangelical Protestant', 'Black Protestant',
                  'Catholic', 'Jewish', 'No religion', 'Other religion']
data2004['reltrad_f'] = pd.Categorical(
    reltrad_code.map(dict(zip([1, 2, 3, 4, 5, 6, 7], reltrad_levels))),
    categories=reltrad_levels, ordered=False,
)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1669062684.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['reltrad_f'] = pd.Categorical(


### Religion strength

Note: the R code maps `reliten == 1` (strongly affiliated) to code 1 with label "Not strong", and `reliten %in% 2:4` to code 0 with label "Strong". That looks like a label inversion in the R, but the Python preserves it verbatim so downstream tables match.

In [16]:
reliten = data2004['reliten']
religstr_code = pd.Series(np.nan, index=data2004.index)
religstr_code[reliten == 1] = 1
religstr_code[reliten.isin([2, 3, 4])] = 0
data2004['religstr_f'] = pd.Categorical(
    religstr_code.map({0.0: 'Strong', 1.0: 'Not strong'}),
    categories=['Strong', 'Not strong'], ordered=False,
)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/204173312.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['religstr_f'] = pd.Categorical(


### Income

In [17]:
realinc = pd.to_numeric(data2004['realinc'], errors='coerce')
data2004['realinc_n'] = realinc
data2004['realinc2004_n'] = realinc * 1.72
data2004['lnrealinc2004_n'] = np.log(realinc * 1.72)
data2004['realinc2004_f'] = pd.cut(
    realinc,
    bins=[-np.inf, 10000, 20000, 30000, np.inf],
    right=False,
    labels=['< 10k', '10-20k', '20-30k', '> 30k'],
)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1630384166.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['realinc_n'] = realinc
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1630384166.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['realinc2004_n'] = realinc * 1.72
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/1630384166.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whi

### Partisanship

In [18]:
partyid = data2004['partyid']
party_code = pd.Series(np.nan, index=data2004.index)
party_code[partyid == 0] = 1
party_code[partyid.isin([1, 2])] = 2
party_code[partyid == 3] = 3
party_code[partyid.isin([4, 5])] = 4
party_code[partyid == 6] = 5
# partyid == 7 falls through to NA

party_levels = ['Strong Democrat', 'Democrat', 'Independent', 'Republican', 'Strong Republican']
data2004['party_f'] = pd.Categorical(
    party_code.map(dict(zip([1, 2, 3, 4, 5], party_levels))),
    categories=party_levels, ordered=False,
)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/308657778.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['party_f'] = pd.Categorical(


### Born in the US

In [19]:
born = data2004['born']
born_code = pd.Series(np.nan, index=data2004.index)
born_code[born == 1] = 0
born_code[born == 2] = 1
data2004['born_usa_f'] = pd.Categorical(
    born_code.map({0.0: 'Born in this country', 1.0: 'Not born in this country'}),
    categories=['Born in this country', 'Not born in this country'], ordered=False,
)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/908040038.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['born_usa_f'] = pd.Categorical(


### Region

In [20]:
region = data2004['region']
region_code = pd.Series(np.nan, index=data2004.index)
region_code[region.isin([1, 2])] = 1
region_code[region.isin([3, 4])] = 2
region_code[region.isin([5, 6, 7])] = 3
region_code[region == 8] = 4
region_code[region == 9] = 5

region_levels = ['Northeast', 'Midwest', 'South', 'Mountain', 'Pacific']
data2004['region_f'] = pd.Categorical(
    region_code.map(dict(zip([1, 2, 3, 4, 5], region_levels))),
    categories=region_levels, ordered=False,
)

/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_52581/796436765.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data2004['region_f'] = pd.Categorical(


## Missing values & subset

In [21]:
base_outcomes = q2_vars + q3_vars + q4_vars + q5_vars
f_outcomes = [v + '_f' for v in base_outcomes]
n_outcomes = [v + '_n' for v in base_outcomes]
all_outcomes = f_outcomes + n_outcomes

f_controls = ['sex_f', 'age_f', 'race_f', 'educ_f', 'born_usa_f',
              'realinc2004_f', 'party_f', 'religstr_f', 'reltrad_f', 'region_f']
n_controls = ['age_n', 'educ_n', 'realinc2004_n', 'lnrealinc2004_n']
all_controls = f_controls + n_controls

all_vars = ['id', 'wgt'] + all_outcomes + all_controls

mask = ~data2004[all_outcomes].isna().all(axis=1)
data2004_recoded = data2004.loc[mask, all_vars].copy()
print(f'after dropping respondents without any Identity-module answer: {data2004_recoded.shape}')

after dropping respondents without any Identity-module answer: (1215, 62)


In [22]:
na_counts = data2004_recoded[f_outcomes].isna().sum()
print(f'Total outcome NAs: {int(na_counts.sum())}')
print(f'Avg outcome NAs per row: {data2004_recoded[f_outcomes].isna().sum(axis=1).mean():.3f}')
print()
print('Outcomes with NAs:')
print(na_counts[na_counts > 0])
print()
print('Controls with NAs:')
ctrl_na = data2004_recoded[all_controls].isna().sum()
print(ctrl_na[ctrl_na > 0])

Total outcome NAs: 741
Avg outcome NAs per row: 0.610

Outcomes with NAs:
clseusa_f     25
ambornin_f    21
amcit_f        9
amlived_f     10
amenglsh_f     2
amchrstn_f    35
amgovt_f       9
amfeel_f      24
amcitizn_f    16
amshamed_f    13
belikeus_f    18
ambetter_f    18
ifwrong_f     22
proudsss_f    48
proudgrp_f    53
proudpol_f    64
prouddem_f    65
proudeco_f    34
proudspt_f    73
proudart_f    81
proudhis_f    31
proudmil_f    31
proudsci_f    39
dtype: int64

Controls with NAs:
age_f                1
born_usa_f           1
realinc2004_f      108
party_f             13
religstr_f          15
age_n                1
realinc2004_n      108
lnrealinc2004_n    108
dtype: int64


## kNN imputation

Pragmatic Python equivalent of `VIM::kNN(k=3)`: ordinal-encode categoricals, run `sklearn.impute.KNNImputer(n_neighbors=3)` on the full numeric matrix, then decode categoricals via nearest-integer.

In [23]:
def knn_impute_mixed(df, n_neighbors=3):
    enc = pd.DataFrame(index=df.index)
    cat_meta = {}
    passthrough = {}
    for c in df.columns:
        col = df[c]
        if hasattr(col, 'cat'):
            cats = list(col.cat.categories)
            cat_meta[c] = cats
            enc[c] = col.cat.codes.where(col.notna()).astype(float)
        elif is_numeric_dtype(col):
            enc[c] = pd.to_numeric(col, errors='coerce')
        else:
            passthrough[c] = col

    imputer = KNNImputer(n_neighbors=n_neighbors)
    imputed_arr = imputer.fit_transform(enc.values)
    imputed = pd.DataFrame(imputed_arr, columns=enc.columns, index=enc.index)

    out = df.copy()
    for c in enc.columns:
        if c in cat_meta:
            cats = cat_meta[c]
            codes = np.clip(np.round(imputed[c]).astype(int), 0, len(cats) - 1)
            out[c] = pd.Categorical([cats[i] for i in codes], categories=cats, ordered=False)
        else:
            out[c] = imputed[c].values
    for c, col in passthrough.items():
        out[c] = col.values
    return out

## Write per-approach parquets

In [24]:
if approach == 'replic':
    misclass_ids = {769, 1523, 2581, 2745}

    keep_1077 = (
        (data2004_recoded[f_outcomes].isna().sum(axis=1) < 10)
        & (data2004_recoded[all_controls].isna().sum(axis=1) == 0)
        & (~data2004_recoded['id'].isin(misclass_ids))
    )
    data2004_1077_ni = data2004_recoded[keep_1077].copy()
    data2004_1077 = knn_impute_mixed(data2004_1077_ni, n_neighbors=3)
    print(f'1077 rows: {len(data2004_1077)}')
    data2004_1077_ni.to_parquet(f'{OUT_DIR}/data2004_1077_ni.parquet')
    data2004_1077.to_parquet(f'{OUT_DIR}/data2004_1077.parquet')

    keep_830 = (
        (data2004_recoded[f_outcomes].isna().sum(axis=1) == 0)
        & (data2004_recoded[all_controls].isna().sum(axis=1) == 0)
        & (~data2004_recoded['id'].isin(misclass_ids))
    )
    data2004_830_ni = data2004_recoded[keep_830].copy()
    data2004_830 = knn_impute_mixed(data2004_830_ni, n_neighbors=3)
    print(f'830 rows: {len(data2004_830)}')
    data2004_830_ni.to_parquet(f'{OUT_DIR}/data2004_830_ni.parquet')
    data2004_830.to_parquet(f'{OUT_DIR}/data2004_830.parquet')

1077 rows: 1077
830 rows: 830


In [25]:
if approach == 'own':
    data2004_1215_ni = data2004_recoded[data2004_recoded['id'] != 769].copy()
    data2004_1215 = knn_impute_mixed(data2004_1215_ni, n_neighbors=3)
    print(f'1215 rows: {len(data2004_1215)}')
    data2004_1215.to_parquet(f'{OUT_DIR}/data2004_1215.parquet')